In [ ]:
%pip uninstall tensorflow
%pip install tensorflow-gpu

In [1]:

# Check for GPU availability
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
tf.debugging.set_log_device_placement(True)

# Ensure TensorFlow uses the GPU if available
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU is configured and ready for use.")
else:
    print("No GPU detected. Training will use CPU.")


Num GPUs Available:  0
No GPU detected. Training will use CPU.


In [ ]:
%pip install keras

In [ ]:
%pip install tensorflow

In [9]:

import numpy as np 
import pandas as pd
import math

from keras.models import Model
from keras.layers import Input, LSTM, Dense,TimeDistributed, Bidirectional

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from keras.utils import to_categorical,Sequence
from keras.optimizers import Adam
from keras.callbacks import TensorBoard,ModelCheckpoint

from keras import backend  as K
import tensorflow as tf



In [ ]:
df = pd.read_csv(r"H:\\THESIS\\DATASET\\2022-08-03-ss.cleaned.csv").sort_values(by=['pdb_id','chain_code'])
print('number of rows (sequences): {}'.format(df.shape[0]))
print('number of unique proteins: {}'.format(len(df.pdb_id.unique())))
df.head()

In [ ]:
gb = df.groupby(['pdb_id'],as_index=False)['chain_code'].count().sort_values(by='chain_code',ascending=False)
gb.rename(columns={'chain_code':'number of chains'},inplace=True)
gb2 = gb.groupby('number of chains',as_index=False)['pdb_id'].count().sort_values(by='pdb_id',ascending=False)

gb3 = gb2.rename(columns={'pdb_id':'proteins count'})
gb3['percentage of proteins'] = gb3['proteins count']/len(df.pdb_id.unique())
gb3.plot.bar(x='number of chains',y='percentage of proteins',figsize=(10,4))

In [ ]:
a = df.loc[df.pdb_id=='1A30'].sort_values(by='chain_code')
a

In [ ]:
print('Chain {} Sequence:\n{}'.format('A',a.loc[a.chain_code=='A','seq'].values[0]))
print('Chain {} Sequence:\n{}'.format('B',a.loc[a.chain_code=='B','seq'].values[0]))

In [ ]:
print('Chain {} Q8:\n{}'.format('A',a.loc[a.chain_code=='A','sst8'].values[0]))
print('Chain {} Q8:\n{}'.format('B',a.loc[a.chain_code=='B','sst8'].values[0]))

In [ ]:
print('Chain {} Q3:\n{}'.format('A',a.loc[a.chain_code=='A','sst3'].values[0]))
print('Chain {} Q3:\n{}'.format('B',a.loc[a.chain_code=='B','sst3'].values[0]))

In [ ]:
# most popular sequences
gb = df.groupby('seq',as_index=False)['chain_code'].count().sort_values(by='chain_code',ascending=False)
gb.rename(columns={'chain_code':'sequence occurence'},inplace=True)
gb[['sequence occurence']].describe()


## B. Predicting the secondary structure of a chain

#### Data Preprocessing Steps


1.    Subset only **unique pairs** of sequences and Q3 secondary structures (sst3) to avoid redundancy when evaluating the model (remember that a given sequence can have multiple sst3 within or across proteins.
2.   Split the data into training/validation/test sets.

3.   Convert the sequences to numerical format before we can feed them to a model.

I have compute constraints,  I will limit the datasets to sequences no longer than 128.

In [ ]:


# subset unique pairs seq/sst3 and exclude 
max_len = 128
df = df.loc[(df.len <= max_len)]

print('There are {} seq with length 128 or less.'.format(df.shape[0]))
df = df.loc[:,['seq','sst3']].drop_duplicates()
print('There are {} unique seq / sst3 pairs.'.format(df.shape[0]))

In [ ]:
## split data into train/val/test
RS = 15141312
np.random.seed(RS)

val_size=.10
test_size=.10

df['split'] = np.random.rand(len(df))

df_val = df.loc[df.split < val_size]
df_test = df.loc[(df.split < val_size + test_size) & (df.split >= val_size)]
df_train = df.loc[df.split >= val_size + test_size]

print(df_train.shape,df_test.shape,df_val.shape)

In [37]:
# preprocesing input sequence and sst3
input_tk = Tokenizer(char_level=True,lower=False)
input_tk.fit_on_texts(df.seq)

target_tk = Tokenizer(char_level=True,lower=False)
target_tk.fit_on_texts(df.sst3)

# char to index
input_char_index = input_tk.word_index
target_char_index = target_tk.word_index

# index to char
input_index_char = dict((i, char) for char, i in input_char_index.items())
target_index_char = dict((i, char) for char, i in target_char_index.items())

# number of tokens (including padding)
num_input_tokens = 1 + len(input_char_index.items())
num_target_tokens = 1 + len(target_char_index.items())

# vectorizing
def trans_seq_in(iseq):
    maxlen = max([len(txt) for txt in iseq])
    input_seq = input_tk.texts_to_sequences(iseq)
    input_seq = pad_sequences(input_seq,maxlen=maxlen,padding='post',value=0)
    return to_categorical(input_seq,num_classes=num_input_tokens)

def trans_seq_out(iseq):
    maxlen = max([len(txt) for txt in iseq])
    target_seq = target_tk.texts_to_sequences(iseq)
    target_seq = pad_sequences(target_seq,maxlen=maxlen,padding='post',value=0)
    return to_categorical(target_seq,num_classes=num_target_tokens)

In [38]:
# generate training and testing data,labels pairs
def ProteinV(data):
    x = trans_seq_in(data.seq)
    y = trans_seq_out(data.sst3)
    return x, y
   
x_train,y_train = ProteinV(df_train)
x_val,y_val = ProteinV(df_val)
x_test,y_test = ProteinV(df_test)

In [ ]:
x_train.shape,y_train.shape,x_val.shape,y_val.shape,x_test.shape,y_test.shape

In [ ]:
# set up model.
x_in = Input(shape=(None, num_input_tokens))
x = Bidirectional(LSTM(100, return_sequences=True))(x_in)
x = TimeDistributed(Dense(100, activation='relu'))(x)
x_out = TimeDistributed(Dense(num_target_tokens, activation='softmax'))(x)

mdl= Model(x_in,x_out)
mdl.summary()

In [47]:


def q3_acc(y_true, y_pred):
    # Convert y_pred from one-hot encoding to class indices
    y_pred_labels = tf.argmax(y_pred, axis=-1)
    # Ensure y_true is also in class indices
    y_true_labels = tf.cast(tf.argmax(y_true, axis=-1), tf.int32) if y_true.shape.ndims == 3 else tf.cast(y_true, tf.int32)
    # Mask to ignore padding tokens (assuming padding tokens are 0)
    mask = tf.greater(y_true_labels, 0)
    # Ensure both tensors are int32 for comparison
    y_true_labels = tf.cast(y_true_labels, tf.int32)
    y_pred_labels = tf.cast(y_pred_labels, tf.int32)
    # Compare masked values
    correct_preds = tf.boolean_mask(y_true_labels, mask) == tf.boolean_mask(y_pred_labels, mask)
    # Return accuracy
    return tf.reduce_mean(tf.cast(correct_preds, tf.float32))




In [ ]:
# compile and train
mdl.compile(optimizer='adam', loss='categorical_crossentropy', metrics=[q3_acc])
mdl.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=40, batch_size=64)
